## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-02-17-53-51 +0000,startribune,Live: Minnesota sets new record for early voting,https://www.startribune.com/live-minnesota-202...
1,2026-11-02-17-50-45 +0000,chitri,Aldermen put $75 million in parking meter mone...,https://www.chicagotribune.com/2026/11/02/alde...
2,2026-11-02-17-49-49 +0000,nyt,Clancy Hearing Focuses on Holdout Juror’s Incl...,https://www.nytimes.com/2026/11/02/us/lindsay-...
3,2026-11-02-17-48-42 +0000,nyt,Trump’s Tariffs Are On the Ballot in Michigan,https://www.nytimes.com/2026/11/02/business/ec...
4,2026-11-02-17-47-00 +0000,wsj,The balance of power in the Senate is on the l...,https://www.wsj.com/politics/elections/senate-...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2438/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
27,trump,25
33,senate,16
50,florida,14
78,election,14
506,race,13


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
214,2026-11-02-05-08-00 +0000,wsj,Rep. Haley Stevens stunned Democrats on Sunday...,https://www.wsj.com/politics/elections/how-rep...,122
289,2026-11-01-18-25-38 +0000,chitri,Democrat Haley Stevens endorses GOP’s Mike Rog...,https://www.chicagotribune.com/2026/11/01/demo...,94
134,2026-11-02-11-55-00 +0000,wsj,As the midterms bumble and stumble and fumble ...,https://www.wsj.com/politics/trumps-monday-spe...,81
9,2026-11-02-17-38-10 +0000,nypost,"Trump takes fight over 20,000 California mail-...",https://nypost.com/2026/11/02/us-news/trump-ta...,76
201,2026-11-02-07-43-25 +0000,nyt,Judge Rejects Trump Officials’ Move to Challen...,https://www.nytimes.com/2026/11/01/us/alaska-s...,75


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
214,122,2026-11-02-05-08-00 +0000,wsj,Rep. Haley Stevens stunned Democrats on Sunday...,https://www.wsj.com/politics/elections/how-rep...
134,81,2026-11-02-11-55-00 +0000,wsj,As the midterms bumble and stumble and fumble ...,https://www.wsj.com/politics/trumps-monday-spe...
276,56,2026-11-01-20-00-00 +0000,wsj,"In the Florida governor’s race, Democrat David...",https://www.wsj.com/politics/elections/democra...
75,53,2026-11-02-15-05-21 +0000,nypost,Two California students left in horrific condi...,https://nypost.com/2026/11/02/us-news/two-cali...
34,44,2026-11-02-16-46-44 +0000,nypost,US and Iran resume ‘very intensive’ peace talk...,https://nypost.com/2026/11/02/world-news/us-an...
9,43,2026-11-02-17-38-10 +0000,nypost,"Trump takes fight over 20,000 California mail-...",https://nypost.com/2026/11/02/us-news/trump-ta...
277,37,2026-11-01-19-54-47 +0000,nyt,Alaska’s Republican senator says he’ll accept ...,https://www.nytimes.com/live/2026/11/01/us/mid...
120,36,2026-11-02-12-35-00 +0000,wsj,The Morning Risk Report: Binance Resumes Withd...,https://www.wsj.com/risk-compliance-journal/th...
49,34,2026-11-02-16-06-02 +0000,startribune,Judge cites 'weaponization' in tossing charges...,https://www.startribune.com/judge-dismisses-co...
133,34,2026-11-02-11-59-01 +0000,nypost,Hackers breach emergency alert apps in Ukraine...,https://nypost.com/2026/11/02/world-news/hacke...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
